# Cleaning the 2018 Taiwan traffic accident data

Source: `accidents_2018.zip` (13 CSV files: one A1 file for fatal accidents, twelve monthly A2 files for injury accidents).

**How the data is shaped:** each row is one *party* (a driver, pedestrian, or object) in an accident. Accident-level fields (date, place, weather, road, cause...) are repeated on every party row. There is **no accident ID**, so we build one.

Run the cells top to bottom. Change `ZIP_PATH` and `OUT_DIR` to match your folders.

In [ ]:
import zipfile, re
import numpy as np
import pandas as pd

ZIP_PATH = 'accidents_2018.zip'      # path to the zip you uploaded
OUT_DIR  = '.'                       # where cleaned files will be saved
pd.set_option('display.max_columns', 60)

## Step 1. Load all 13 CSV files straight from the zip
The files are UTF-8 with a BOM, so we use `encoding='utf-8-sig'`. We keep the source file name so you can trace any row back.

In [ ]:
frames = []
with zipfile.ZipFile(ZIP_PATH) as z:
    data_files = sorted(n for n in z.namelist() if re.search(r'A[12].*\.csv$', n))
    for name in data_files:
        with z.open(name) as f:
            part = pd.read_csv(f, encoding='utf-8-sig', low_memory=False)
        part['source_file'] = name
        frames.append(part)
        print(f'{name}: {part.shape}')

raw = pd.concat(frames, ignore_index=True)
print('combined:', raw.shape)

## Step 2. Rename columns to short English names
The original Chinese headers are long. The mapping is kept in `COLS` so you can always look up the original name.

In [ ]:
COLS = {
    '發生年度': 'year', '發生月份': 'month', '發生日期': 'date', '發生時間': 'time',
    '事故類別名稱': 'accident_class', '處理單位名稱警局層': 'police_dept', '發生地點': 'location',
    '天候名稱': 'weather', '光線名稱': 'light',
    '道路類別-第1當事者-名稱': 'road_class', '速限-第1當事者': 'speed_limit',
    '道路型態大類別名稱': 'road_type', '道路型態子類別名稱': 'road_type_sub',
    '事故位置大類別名稱': 'position', '事故位置子類別名稱': 'position_sub',
    '路面狀況-路面鋪裝名稱': 'surface_pavement', '路面狀況-路面狀態名稱': 'surface_condition',
    '路面狀況-路面缺陷名稱': 'surface_defect',
    '道路障礙-障礙物名稱': 'obstacle', '道路障礙-視距品質名稱': 'sight_quality', '道路障礙-視距名稱': 'sight_distance',
    '號誌-號誌種類名稱': 'signal_type', '號誌-號誌動作名稱': 'signal_action',
    '車道劃分設施-分向設施大類別名稱': 'divider', '車道劃分設施-分向設施子類別名稱': 'divider_sub',
    '車道劃分設施-分道設施-快車道或一般車道間名稱': 'lane_line',
    '車道劃分設施-分道設施-快慢車道間名稱': 'fast_slow_line',
    '車道劃分設施-分道設施-路面邊線名稱': 'edge_line',
    '事故類型及型態大類別名稱': 'crash_type', '事故類型及型態子類別名稱': 'crash_type_sub',
    '肇因研判大類別名稱-主要': 'main_cause', '肇因研判子類別名稱-主要': 'main_cause_sub',
    '死亡受傷人數': 'casualties', '當事者順位': 'party_order',
    '當事者區分-類別-大類別名稱-車種': 'vehicle', '當事者區分-類別-子類別名稱-車種': 'vehicle_sub',
    '當事者屬-性-別名稱': 'sex', '當事者事故發生時年齡': 'age',
    '保護裝備名稱': 'protection', '行動電話或電腦或其他相類功能裝置名稱': 'phone_use',
    '當事者行動狀態大類別名稱': 'action', '當事者行動狀態子類別名稱': 'action_sub',
    '車輛撞擊部位大類別名稱-最初': 'impact_first', '車輛撞擊部位子類別名稱-最初': 'impact_first_sub',
    '車輛撞擊部位大類別名稱-其他': 'impact_other', '車輛撞擊部位子類別名稱-其他': 'impact_other_sub',
    '肇因研判大類別名稱-個別': 'party_cause', '肇因研判子類別名稱-個別': 'party_cause_sub',
    '肇事逃逸類別名稱-是否肇逃': 'hit_and_run', '經度': 'lon', '緯度': 'lat',
}
df = raw.rename(columns=COLS)
assert set(COLS.values()) <= set(df.columns), 'some columns were not renamed'
df.info()

## Step 3. Remove the footer rows
Each file ends with two note lines (`資料提供日期：111年09月28日` = data release date, `事故類別：A2類` = accident class). They land in the `year` column and every other field is empty.

In [ ]:
footer = df['accident_class'].isna()
print(df.loc[footer, 'year'].value_counts())
df = df.loc[~footer].reset_index(drop=True)
print('rows left:', len(df))

## Step 4. Check exact duplicate rows
There are 3 rows that are identical to another row. Looking at them, each sits in a group with **two different party-1 drivers** at the same date, time and place, so they look like two separate accidents recorded at the same spot rather than copy errors. We **keep** them and only flag them.

In [ ]:
dup = df.drop(columns='source_file').duplicated(keep=False)
print('rows involved in exact duplicates:', dup.sum())
df['is_exact_duplicate'] = df.drop(columns='source_file').duplicated()
df.loc[dup, ['date', 'time', 'location', 'party_order', 'vehicle', 'age']]

## Step 5. Fix types: year, month, date, time
* `date` is stored as a float like `20180101.0` and `time` as `51200.0` meaning 05:12:00 (HHMMSS).
* Two rows have the impossible time `18510` (01:85:10). They become missing (`NaT`) instead of being dropped.

In [ ]:
df['year'] = df['year'].astype(int)
df['month'] = df['month'].astype(int)
df['party_order'] = df['party_order'].astype(int)

df['date'] = pd.to_datetime(df['date'].astype(int).astype(str), format='%Y%m%d')

t = df['time'].astype(int).astype(str).str.zfill(6)        # '051200'
df['datetime'] = pd.to_datetime(df['date'].dt.strftime('%Y%m%d') + t,
                                format='%Y%m%d%H%M%S', errors='coerce')
df['hour'] = df['datetime'].dt.hour.astype('Int64')
df['weekday'] = df['date'].dt.day_name()
print('invalid times ->', df['datetime'].isna().sum())
df = df.drop(columns='time')

# sanity check: month column agrees with the date
assert (df['month'] == df['date'].dt.month).all()

## Step 6. Split `casualties` into numbers
`casualties` looks like `死亡1;受傷0` (deaths 1; injuries 0). It is an accident-level value, repeated on every party row.

In [ ]:
cas = df['casualties'].str.extract(r'死亡(\d+);受傷(\d+)').astype(int)
df['deaths'] = cas[0]
df['injuries'] = cas[1]
df = df.drop(columns='casualties')
df[['deaths', 'injuries']].describe()

## Step 7. Build an accident ID
Rows of one accident are **not** always in order (party 3 can come before party 2), so counting `party_order == 1` does not work. Instead we group on all the accident-level columns. This gives 320,312 accidents, and every one has exactly one party 1 except the 3 same-place-same-time cases from Step 4.

In [ ]:
ACC_COLS = ['year', 'month', 'date', 'datetime', 'accident_class', 'police_dept', 'location',
            'weather', 'light', 'road_class', 'speed_limit', 'road_type', 'road_type_sub',
            'position', 'position_sub', 'surface_pavement', 'surface_condition', 'surface_defect',
            'obstacle', 'sight_quality', 'sight_distance', 'signal_type', 'signal_action',
            'divider', 'divider_sub', 'lane_line', 'fast_slow_line', 'edge_line',
            'crash_type', 'crash_type_sub', 'main_cause', 'main_cause_sub',
            'deaths', 'injuries', 'lon', 'lat']
df['accident_id'] = df.groupby(ACC_COLS, dropna=False, sort=False).ngroup()
print('accidents:', df['accident_id'].nunique())
print((df.groupby('accident_id')['party_order'].apply(lambda s: (s == 1).sum())).value_counts())

## Step 8. Clean `age`
What we found:
* `-1` = unknown (21,752 rows).
* A spike at **107** (5,636 rows): 107 is the year 2018 in the Taiwan (ROC) calendar, and almost all of these rows are objects or unfound hit-and-run drivers, so it is a placeholder, not an age.
* Objects (`無或物(動物、堆置物)`, animal/obstacle) and unfound hit-and-run drivers (`肇事逃逸尚未查獲`) have no real age.
* A few values go up to 210.

Rule: age is kept only for people (`sex` is 男 or 女) with 0 ≤ age ≤ 100. Everything else becomes `NaN`. We also add `is_person`.

In [ ]:
df['is_person'] = df['sex'].isin(['男', '女'])
valid_age = df['is_person'] & df['age'].between(0, 100)
print('ages set to NaN:', (~valid_age).sum(), ' of which people:', (df['is_person'] & ~valid_age).sum())
df['age'] = df['age'].where(valid_age).astype('Int64')
df['age'].describe()

## Step 9. Clean `speed_limit`
Most values are 30–110 km/h. There are `0` (3,664 rows), impossible values like 330–707 (typing errors, e.g. 501 for 50), and odd values like 1, 2, 41. Rule: keep 5–110 that are multiples of 5, otherwise `NaN`.

In [ ]:
sl = df['speed_limit']
valid_sl = sl.between(5, 110) & (sl % 5 == 0)
print('speed limits set to NaN:', (~valid_sl).sum())
df['speed_limit'] = sl.where(valid_sl).astype('Int64')
df['speed_limit'].value_counts().sort_index()

## Step 10. Clean coordinates
Taiwan including Kinmen, Matsu and Penghu lies roughly in longitude 118–122.5 and latitude 21.5–26.5.
* 4 rows have longitude and latitude **swapped** (e.g. lon 24.14, lat 120.70). We swap them back.
* 73 rows are clearly wrong (e.g. -180 / 85.05, which is a map default). They become `NaN`. The rows stay, only the coordinates are removed.

In [ ]:
swapped = df['lon'].between(21.5, 26.5) & df['lat'].between(118, 122.5)
df.loc[swapped, ['lon', 'lat']] = df.loc[swapped, ['lat', 'lon']].values
print('swapped back:', swapped.sum())

in_taiwan = df['lon'].between(118, 122.5) & df['lat'].between(21.5, 26.5)
print('coordinates set to NaN:', (~in_taiwan).sum())
df.loc[~in_taiwan, ['lon', 'lat']] = np.nan

## Step 11. Missing values in the party columns
Most missing values are **structural**, not errors:
* `vehicle`, `protection`, `phone_use`, `action`, `impact_first` are empty when the party is an object or an unfound hit-and-run driver (~5% of rows). We leave them `NaN`; use `is_person` to filter.
* `impact_other` is empty in 63% of rows because most vehicles were hit in only one place. We fill it with `無` (none).
* `hit_and_run` is missing in 8 rows. We leave it `NaN`.

In [ ]:
df[['impact_other', 'impact_other_sub']] = df[['impact_other', 'impact_other_sub']].fillna('無')
miss = df.isna().mean().mul(100).round(2)
miss[miss > 0].sort_values(ascending=False)

## Step 12. Tidy text columns and set categories
Strip stray spaces and convert repeated text columns to `category`, which saves a lot of memory.

In [ ]:
text_cols = df.select_dtypes(include='object').columns.drop(['location', 'source_file'])
for c in text_cols:
    df[c] = df[c].str.strip().astype('category')

df['hit_and_run'] = df['hit_and_run'].map({'是': True, '否': False}).astype('boolean')
df['source_file'] = df['source_file'].str.replace('.csv', '', regex=False)

front = ['accident_id', 'party_order', 'accident_class', 'date', 'datetime', 'hour', 'weekday']
df = df[front + [c for c in df.columns if c not in front]]
df = df.sort_values(['date', 'datetime', 'accident_id', 'party_order']).reset_index(drop=True)
df.info(memory_usage='deep')

## Step 13. Build an accident-level table (one row per accident)
Use this for questions about accidents (when, where, how many deaths). Use the party table for questions about people and vehicles.

In [ ]:
party1 = df[df['party_order'] == 1].drop_duplicates('accident_id')
acc_extra = df.groupby('accident_id').agg(n_parties=('party_order', 'size'),
                                          any_hit_and_run=('hit_and_run', 'max'))
keep = ['accident_id'] + [c for c in ACC_COLS if c in df.columns] + ['hour', 'weekday']
accidents = party1[keep].merge(acc_extra, on='accident_id')
# accidents with no party-1 row (none in 2018, kept for safety)
missing = set(df['accident_id']) - set(accidents['accident_id'])
print('accidents:', len(accidents), ' without party 1:', len(missing))
accidents.head()

## Step 14. Save
Saved as gzip-compressed CSV (`utf-8-sig` so Excel shows the Chinese text). Read back with `pd.read_csv(path, parse_dates=['date', 'datetime'])`.

In [ ]:
import os
os.makedirs(OUT_DIR, exist_ok=True)
df.to_csv(f'{OUT_DIR}/accidents_2018_parties_clean.csv.gz', index=False, encoding='utf-8-sig')
accidents.to_csv(f'{OUT_DIR}/accidents_2018_accidents_clean.csv.gz', index=False, encoding='utf-8-sig')
print(df.shape, accidents.shape)